# 🏥 Lab Session: Building a Transformer-based Text Classifier
## Healthcare Medical News & Symptom Classification using BERT

---

### 🎯 Project Overview

**Industry Domain:** Healthcare  
**Model:** `bert-base-uncased` (Google, 2018)  
**Data Source:** Live scraping from [MedlinePlus Health News](https://medlineplus.gov/news/) (US National Library of Medicine)

**What we're building:**  
A real-world text classifier that reads healthcare news headlines + summaries scraped from the web and classifies them into medical categories like *Heart Disease*, *Diabetes*, *Mental Health*, *Cancer*, *Nutrition*, etc.

---

### 🗺️ Notebook Flow

```
Step 1 → Install & Import Libraries
Step 2 → Scrape Healthcare News from MedlinePlus
Step 3 → Explore & Prepare the Dataset
Step 4 → Tokenization using BERT Tokenizer
Step 5 → Build PyTorch Dataset & DataLoaders
Step 6 → Load BERT Model for Classification
Step 7 → Train the Model
Step 8 → Evaluate the Model
Step 9 → Predict on New Custom Text
```

---

### 📌 Prerequisites
- Python 3.8+
- Basic understanding of: ANN, RNN, Transformer architecture
- Familiarity with: PyTorch basics, pandas, sklearn

## Step 1: Install & Import Libraries

We need:
- `transformers` — Hugging Face library for BERT
- `torch` — PyTorch for model training
- `beautifulsoup4` + `requests` — Web scraping
- `scikit-learn` — Evaluation metrics
- `pandas`, `numpy` — Data handling

In [ ]:
# ── Install required packages (run once) ──────────────────────────────
!pip install transformers torch scikit-learn beautifulsoup4 requests pandas numpy tqdm -q

In [ ]:
# ── Core imports ───────────────────────────────────────────────────────
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import re
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# Hugging Face Transformers
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ── Device setup ───────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Libraries loaded successfully")
print(f"🖥️  Using device: {device}")

## Step 2: Scrape Healthcare News from MedlinePlus

**Data Source:** [MedlinePlus](https://medlineplus.gov) — a service of the US National Library of Medicine (NLM).  
It provides reliable, publicly accessible health news articles organized by topic.

**Strategy:**
- Scrape headlines + summaries from multiple topic pages
- Each topic page = one category label
- Categories: Heart Disease, Diabetes, Mental Health, Cancer, Nutrition

```
MedlinePlus Topic Page
       ↓
  [Headline 1] + [Summary 1]  →  Label: "Heart Disease"
  [Headline 2] + [Summary 2]  →  Label: "Heart Disease"
       ...
```

In [ ]:
# ── Category → MedlinePlus URL mapping ────────────────────────────────
CATEGORIES = {
    "Heart Disease" : "https://medlineplus.gov/heartdiseases.html",
    "Diabetes"      : "https://medlineplus.gov/diabetes.html",
    "Mental Health" : "https://medlineplus.gov/mentalhealth.html",
    "Cancer"        : "https://medlineplus.gov/cancer.html",
    "Nutrition"     : "https://medlineplus.gov/nutrition.html",
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Educational Research Bot)'
}

def scrape_medlineplus(url, category, max_articles=40):
    """
    Scrapes health article headlines and summaries from a MedlinePlus topic page.
    Returns a list of dicts: {text, category}
    """
    articles = []
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # MedlinePlus news items appear in <ul> with class 'ul-links-news'
        # and also in article summary sections — we grab both
        seen = set()

        # ── Method 1: News link items ──────────────────────────────────
        news_items = soup.select('ul.ul-links-news li a, #section-news li a')
        for item in news_items:
            text = item.get_text(strip=True)
            if len(text) > 30 and text not in seen:
                articles.append({'text': text, 'category': category})
                seen.add(text)

        # ── Method 2: Summary paragraphs & headers ─────────────────────
        summary_items = soup.select(
            '#section-summary p, #section-summary li, '
            '.section-body p, .topic-summary p'
        )
        for item in summary_items:
            text = item.get_text(strip=True)
            text = re.sub(r'\s+', ' ', text)
            if len(text) > 50 and text not in seen:
                articles.append({'text': text[:300], 'category': category})
                seen.add(text)

        # ── Method 3: Any <li> or <a> inside known content divs ────────
        fallback = soup.select('div#topic-summary li, div.section li')
        for item in fallback:
            text = item.get_text(strip=True)
            if len(text) > 40 and text not in seen:
                articles.append({'text': text[:300], 'category': category})
                seen.add(text)

    except Exception as e:
        print(f"  ⚠️  Could not scrape {url}: {e}")

    return articles[:max_articles]


# ── Scrape all categories ──────────────────────────────────────────────
print("🌐 Scraping MedlinePlus health topics...\n")
all_articles = []

for category, url in CATEGORIES.items():
    articles = scrape_medlineplus(url, category, max_articles=40)
    all_articles.extend(articles)
    print(f"  ✅  {category:20s} → {len(articles):3d} articles scraped")
    time.sleep(1)   # polite delay between requests

print(f"\n📦 Total raw articles scraped: {len(all_articles)}")

## Step 3: Explore & Prepare the Dataset

Before training, we need to:
1. Build a DataFrame and inspect the data
2. Supplement with curated seed samples (ensures class balance even if scraping varies)
3. Clean and label-encode the categories

In [ ]:
# ── Curated seed samples (domain-expert backup) ────────────────────────
# These ensure each class has representative samples regardless of
# what was successfully scraped on the day of the session

SEED_SAMPLES = [
    # Heart Disease
    {"text": "High blood pressure increases the risk of heart attack and stroke significantly.", "category": "Heart Disease"},
    {"text": "Cholesterol plaques in coronary arteries can lead to chest pain and cardiac events.", "category": "Heart Disease"},
    {"text": "Atrial fibrillation is an irregular heartbeat that raises risk of blood clots.", "category": "Heart Disease"},
    {"text": "Cardiologists recommend regular ECG screening for patients over 50 years old.", "category": "Heart Disease"},
    {"text": "Heart failure occurs when the heart muscle cannot pump enough blood to meet the body needs.", "category": "Heart Disease"},
    {"text": "Bypass surgery restores blood flow to blocked coronary arteries in severe cases.", "category": "Heart Disease"},
    {"text": "Statins are prescribed to lower LDL cholesterol and reduce cardiovascular disease risk.", "category": "Heart Disease"},
    {"text": "A myocardial infarction happens when blood supply to part of the heart is cut off.", "category": "Heart Disease"},
    {"text": "Lifestyle changes including diet exercise and quitting smoking benefit heart health.", "category": "Heart Disease"},
    {"text": "Echocardiograms use ultrasound waves to create images of heart structure and function.", "category": "Heart Disease"},
    # Diabetes
    {"text": "Type 2 diabetes develops when cells become resistant to insulin and blood sugar rises.", "category": "Diabetes"},
    {"text": "Continuous glucose monitors help diabetic patients track blood sugar in real time.", "category": "Diabetes"},
    {"text": "Metformin is the first-line medication prescribed for managing type 2 diabetes.", "category": "Diabetes"},
    {"text": "Uncontrolled diabetes can lead to kidney damage nerve damage and vision loss.", "category": "Diabetes"},
    {"text": "HbA1c test measures average blood glucose levels over the past two to three months.", "category": "Diabetes"},
    {"text": "Insulin therapy is essential for patients with type 1 diabetes to survive.", "category": "Diabetes"},
    {"text": "Gestational diabetes occurs during pregnancy and can affect both mother and baby.", "category": "Diabetes"},
    {"text": "Diabetic retinopathy is a leading cause of blindness in working-age adults worldwide.", "category": "Diabetes"},
    {"text": "Regular foot care is critical for diabetic patients to prevent ulcers and amputations.", "category": "Diabetes"},
    {"text": "Low carbohydrate diets have shown promise in improving insulin sensitivity in diabetics.", "category": "Diabetes"},
    # Mental Health
    {"text": "Depression affects millions worldwide and is characterized by persistent sadness and loss of interest.", "category": "Mental Health"},
    {"text": "Cognitive behavioral therapy has strong evidence for treating anxiety and depression disorders.", "category": "Mental Health"},
    {"text": "Antidepressants like SSRIs work by increasing serotonin availability in the brain.", "category": "Mental Health"},
    {"text": "Post-traumatic stress disorder can develop after experiencing or witnessing a traumatic event.", "category": "Mental Health"},
    {"text": "Mindfulness meditation practice can reduce symptoms of anxiety and improve emotional regulation.", "category": "Mental Health"},
    {"text": "Schizophrenia is a severe mental disorder that affects how a person thinks feels and behaves.", "category": "Mental Health"},
    {"text": "Bipolar disorder involves extreme mood swings from depression to mania and hypomania.", "category": "Mental Health"},
    {"text": "Early intervention for mental health disorders leads to significantly better long-term outcomes.", "category": "Mental Health"},
    {"text": "Social isolation and loneliness are strongly linked to increased risk of mental illness.", "category": "Mental Health"},
    {"text": "Teletherapy and digital mental health apps have expanded access to psychological support.", "category": "Mental Health"},
    # Cancer
    {"text": "Chemotherapy targets rapidly dividing cells but also affects healthy tissue causing side effects.", "category": "Cancer"},
    {"text": "Breast cancer screening with mammography can detect tumors before symptoms appear.", "category": "Cancer"},
    {"text": "Immunotherapy harnesses the immune system to recognize and destroy cancer cells.", "category": "Cancer"},
    {"text": "Lung cancer is the leading cause of cancer death worldwide and is strongly linked to smoking.", "category": "Cancer"},
    {"text": "Prostate specific antigen PSA testing is used to screen for prostate cancer in men.", "category": "Cancer"},
    {"text": "Genetic mutations in BRCA1 and BRCA2 genes significantly increase breast cancer risk.", "category": "Cancer"},
    {"text": "Radiation therapy uses high energy beams to kill cancer cells and shrink tumors.", "category": "Cancer"},
    {"text": "Colorectal cancer rates are rising among younger adults under 50 years old.", "category": "Cancer"},
    {"text": "Targeted therapy drugs block specific molecules involved in cancer cell growth and spread.", "category": "Cancer"},
    {"text": "Palliative care focuses on providing relief from symptoms and improving quality of life.", "category": "Cancer"},
    # Nutrition
    {"text": "A Mediterranean diet rich in olive oil fish and vegetables reduces chronic disease risk.", "category": "Nutrition"},
    {"text": "Excessive sugar consumption is linked to obesity type 2 diabetes and fatty liver disease.", "category": "Nutrition"},
    {"text": "Omega-3 fatty acids found in fish and flaxseed support brain and cardiovascular health.", "category": "Nutrition"},
    {"text": "Vitamin D deficiency is associated with bone loss immune dysfunction and depression.", "category": "Nutrition"},
    {"text": "Dietary fiber from whole grains legumes and vegetables promotes gut microbiome health.", "category": "Nutrition"},
    {"text": "Ultra-processed foods high in sodium and trans fats increase cardiovascular disease risk.", "category": "Nutrition"},
    {"text": "Iron deficiency anemia is the most common nutritional deficiency globally affecting billions.", "category": "Nutrition"},
    {"text": "Antioxidants in fruits and vegetables protect cells from oxidative stress and inflammation.", "category": "Nutrition"},
    {"text": "Intermittent fasting has shown potential benefits for weight loss and metabolic health.", "category": "Nutrition"},
    {"text": "Adequate protein intake is essential for muscle repair immune function and hormone production.", "category": "Nutrition"},
]

print(f"📌 Curated seed samples ready: {len(SEED_SAMPLES)}")

In [ ]:
# ── Combine scraped + seed data ────────────────────────────────────────
df_scraped = pd.DataFrame(all_articles)
df_seed    = pd.DataFrame(SEED_SAMPLES)
df         = pd.concat([df_scraped, df_seed], ignore_index=True)

# ── Basic cleaning ─────────────────────────────────────────────────────
df['text'] = df['text'].str.strip()
df = df[df['text'].str.len() > 30]   # remove very short entries
df = df.drop_duplicates(subset='text').reset_index(drop=True)

print("📊 Dataset Summary")
print("=" * 40)
print(f"Total samples : {len(df)}")
print(f"Scraped        : {len(df_scraped)}")
print(f"Seed samples   : {len(df_seed)}")
print()
print("Samples per category:")
print(df['category'].value_counts())

In [ ]:
# ── Visualize class distribution ───────────────────────────────────────
plt.figure(figsize=(9, 4))
counts = df['category'].value_counts()
colors = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']
bars = plt.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontweight='bold')
plt.title('Healthcare Dataset — Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Number of Samples')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# ── Preview some samples ───────────────────────────────────────────────
print("\n📋 Sample rows:")
df[['text','category']].sample(6, random_state=42).to_string(index=False, max_colwidth=80)
df[['text','category']].sample(6, random_state=42)

In [ ]:
# ── Label Encoding ─────────────────────────────────────────────────────
# BERT needs integer labels: Heart Disease→0, Diabetes→1, etc.

le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

NUM_CLASSES = len(le.classes_)
LABEL_MAP   = dict(enumerate(le.classes_))

print("🏷️  Label Encoding Map:")
for idx, name in LABEL_MAP.items():
    print(f"   {idx} → {name}")

# ── Train / Validation / Test split ───────────────────────────────────
X = df['text'].tolist()
y = df['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"\n✂️  Data Split:")
print(f"   Train      : {len(X_train)} samples (70%)")
print(f"   Validation : {len(X_val)} samples (15%)")
print(f"   Test       : {len(X_test)} samples (15%)")

## Step 4: Tokenization using BERT Tokenizer

BERT cannot read raw text — it needs tokens converted to integer IDs in a very specific format.

### What `bert-base-uncased` means
- **`base`** → 12 Transformer layers, 768 hidden dimensions, 110M parameters (vs `large` = 24 layers, 1024 dims, 340M params)
- **`uncased`** → ALL text is lowercased before tokenizing. "Doctor" and "doctor" → same tokens. Use `bert-base-cased` when capitalisation carries meaning (e.g., Named Entity Recognition where "Apple" the company ≠ "apple" the fruit).

### BERT's Special Tokens

| Token | ID | Purpose |
|---|---|---|
| `[CLS]` | 101 | **Always first.** BERT's final hidden state at this position is used as the sentence representation for classification. |
| `[SEP]` | 102 | **Separator.** Marks the end of a sentence (or boundary between two sentences in pair tasks). |
| `[PAD]` | 0 | **Padding.** Fills shorter sequences to `MAX_LEN` so all sequences in a batch are the same length. |
| `[UNK]` | 100 | Unknown token — for characters outside BERT's vocabulary (rare with WordPiece). |
| `[MASK]` | 103 | Used only during BERT **pre-training** (Masked Language Modeling) — not used during fine-tuning. |

### Three Inputs BERT Expects

```
Raw text: "High blood pressure increases risk"
              ↓  BertTokenizer

input_ids      : [101, 2152, 2668, 3619,  ...  , 3891, 102,  0,  0]
                  ↑CLS  high blood pressure      risk  ↑SEP  ↑PAD ↑PAD

attention_mask : [  1,    1,    1,    1,  ...  ,    1,   1,  0,  0]
                  ↑real tokens = 1                      ↑padding = 0

token_type_ids : [  0,    0,    0,    0,  ...  ,    0,   0,  0,  0]
                  ↑all 0 for single-sentence tasks
                  (would be 1 for tokens in sentence B in pair tasks like NLI)
```

**Why `attention_mask` matters:** BERT's self-attention runs over all 128 positions. Without the mask, padding tokens (meaningless zeros) would pollute the attention scores. The mask tells BERT: "ignore positions where mask=0".

In [ ]:
# ── Load BERT tokenizer ────────────────────────────────────────────────
# BertTokenizer uses WordPiece algorithm (not BPE like GPT).
# Vocabulary size: 30,522 tokens for bert-base-uncased.
# It lowercases ALL text before tokenizing (that's what "uncased" means).
MODEL_NAME   = 'bert-base-uncased'
MAX_LEN      = 128    # max token length per sample (BERT supports up to 512)
                      # 128 is sufficient for short sentences; use 256-512 for long docs

print(f"⏳ Loading tokenizer: {MODEL_NAME} ...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded  |  Vocab size: {tokenizer.vocab_size:,}")

# ── Demonstrate tokenization on one example ────────────────────────────
sample_text = "High blood pressure increases the risk of heart attack and stroke."

# tokenizer() handles: lowercase → WordPiece split → add [CLS]/[SEP] → pad → convert to IDs
encoded = tokenizer(
    sample_text,
    max_length=MAX_LEN,
    padding='max_length',   # pad shorter sequences to MAX_LEN with [PAD]=0
    truncation=True,         # cut sequences longer than MAX_LEN
    return_tensors='pt'      # return PyTorch tensors (not plain Python lists)
)

tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0][:15].tolist())

print("\n📝 Tokenization Demo:")
print(f"   Original text      : {sample_text}")
print(f"   Token IDs (first 15): {encoded['input_ids'][0][:15].tolist()}")
print(f"   Decoded tokens     : {tokens}")
print(f"   Attention mask     : {encoded['attention_mask'][0][:15].tolist()}")
print(f"   Token type IDs     : {encoded['token_type_ids'][0][:15].tolist()}")

# ── Key observations to verify ─────────────────────────────────────────
print("\n🔍 Key observations:")
print(f"   tokens[0]  = '{tokens[0]}'  → [CLS] token, ID=101  (always first)")
print(f"   tokens[-1] = '{tokenizer.convert_ids_to_tokens([encoded[\"input_ids\"][0][-1].item()])[0]}'  → should be [PAD] at end of padded sequence")
print(f"   [SEP] ID=102 appears at the last real token position")
print(f"   All token_type_ids = 0  (single sentence; would be 1 for sentence B in pair tasks)")
print(f"\n   Total tokens in sequence : {(encoded['attention_mask'][0] == 1).sum().item()} real + "
      f"{(encoded['attention_mask'][0] == 0).sum().item()} padding = {MAX_LEN} total")

# ── WordPiece splitting demo ───────────────────────────────────────────
# WordPiece splits rare/unknown words into sub-word pieces marked with ##
print("\n📌 WordPiece sub-word splitting:")
rare_words = ["cardiomyopathy", "immunosuppressant", "hypoglycemia"]
for word in rare_words:
    pieces = tokenizer.tokenize(word)
    print(f"   '{word}' → {pieces}")

## Step 5: Build PyTorch Dataset & DataLoaders

We wrap our data in a custom `Dataset` class so PyTorch can feed mini-batches to BERT during training.

```
List of texts
     ↓  HealthcareDataset (PyTorch Dataset)
     ↓  DataLoader (batching + shuffling)
 Batch of tensors → BERT model
```

In [ ]:
class HealthcareDataset(Dataset):
    """
    Custom PyTorch Dataset for healthcare text classification.

    PyTorch requires a Dataset subclass with three methods:
      __init__  : store the raw data
      __len__   : return total number of samples (used by DataLoader to know when epoch ends)
      __getitem__: return ONE sample as tensors — DataLoader calls this per index and batches results

    In Java terms: this is like implementing Iterable<Tensor> — DataLoader is the for-loop.
    """
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = self.labels[idx]

        # tokenizer returns shape (1, MAX_LEN) because return_tensors='pt' adds a batch dim.
        # squeeze(0) removes that extra dim → shape becomes (MAX_LEN,)
        # We do NOT want the batch dim here because DataLoader will stack individual
        # samples into batches itself: N × (MAX_LEN,) → (N, MAX_LEN)
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',    # all sequences padded to same length → necessary for batching
            truncation=True,
            return_tensors='pt'
        )

        return {
            # input_ids: integer token IDs — shape (MAX_LEN,) after squeeze
            'input_ids'      : encoding['input_ids'].squeeze(0),

            # attention_mask: 1 for real tokens, 0 for [PAD] tokens — shape (MAX_LEN,)
            # BERT's self-attention uses this to ignore padding positions
            'attention_mask' : encoding['attention_mask'].squeeze(0),

            # label: single integer (0-4) — dtype=long required by CrossEntropyLoss
            'label'          : torch.tensor(label, dtype=torch.long)
        }
        # NOTE: We don't return token_type_ids here because all our inputs are
        # single sentences (not sentence pairs). token_type_ids would all be 0,
        # which is BERT's default when not supplied.


# ── Create Dataset objects ─────────────────────────────────────────────
BATCH_SIZE = 16   # number of samples per forward pass
                  # larger = faster training but more GPU memory
                  # 16 is safe for BERT-base on most GPUs

train_dataset = HealthcareDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = HealthcareDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = HealthcareDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# ── Create DataLoaders ─────────────────────────────────────────────────
# DataLoader wraps Dataset and handles:
#   - Batching  : groups __getitem__ outputs into batches
#   - Shuffling : randomises order each epoch (train only — never shuffle val/test)
#   - Collation : stacks individual dicts into a batch dict of tensors
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Datasets created")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val batches   : {len(val_loader)}")
print(f"   Test batches  : {len(test_loader)}")

# ── Inspect one batch ──────────────────────────────────────────────────
# Shapes should be (BATCH_SIZE, MAX_LEN) — confirms DataLoader stacked correctly
sample_batch = next(iter(train_loader))
print(f"\n🔍 One batch shape:")
print(f"   input_ids      : {sample_batch['input_ids'].shape}     ← (batch_size, MAX_LEN)")
print(f"   attention_mask : {sample_batch['attention_mask'].shape} ← (batch_size, MAX_LEN)")
print(f"   labels         : {sample_batch['label'].shape}           ← (batch_size,)")

## Step 6: Load BERT Model for Sequence Classification

`BertForSequenceClassification` = pre-trained BERT body + a classification head bolted on top.

### What the model looks like internally

```
Input tokens (batch_size × MAX_LEN)
         ↓
  ┌─────────────────────────────────────────┐
  │  BERT Body (pre-trained, ~109M params)   │
  │  12 × [Self-Attention + FFN + LayerNorm] │
  └──────────────────┬──────────────────────┘
                     ↓  output shape: (batch, MAX_LEN, 768)
                     │  one 768-dim vector per token position
                     │
        Extract [CLS] token at position 0
                     │  shape: (batch, 768)
                     ↓
             Pooler Layer (Dense 768→768 + tanh)
                     │  shape: (batch, 768)
                     ↓
               Dropout (p=0.1)   ← regularisation during fine-tuning
                     ↓
     Classification Head: Linear(768 → num_classes)   ← ~4K new params
                     ↓
              Logits (batch × num_classes)
                     ↓
           Softmax → Predicted Category

NOTE: The [CLS] token's final representation is used as the
"sentence embedding" for classification. This works because
during pre-training BERT was trained to pack sentence-level
meaning into the [CLS] position via the NSP task.
```

**Fine-tuning strategy:** Full fine-tuning — ALL BERT weights + classification head are updated during training (not just the head). This gives the best accuracy when you have enough data (100+ samples per class).

In [ ]:
# ── Load pre-trained BERT with classification head ─────────────────────
# from_pretrained downloads weights from Hugging Face Hub (cached locally after first run)
# num_labels tells HF to add Linear(768 → num_labels) as the classification head
print(f"⏳ Loading {MODEL_NAME} for sequence classification...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,         # adds Linear(768 → NUM_CLASSES) classification head
    output_attentions=False,        # don't return attention matrices (saves memory)
    output_hidden_states=False      # don't return all 12 layer outputs (saves memory)
    # Set output_attentions=True if you want to visualise which tokens the model attends to
)
model = model.to(device)   # move ALL parameters to GPU (if available) or CPU

# ── Model summary ──────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# All params are trainable because we're doing full fine-tuning.
# If you wanted frozen BERT, you'd do:
#   for param in model.bert.parameters(): param.requires_grad = False
# Then only the classification head would train (~4K params vs 110M).

print(f"\n✅ BERT model loaded")
print(f"   Architecture         : bert-base-uncased")
print(f"   Transformer layers   : 12")
print(f"   Attention heads      : 12 per layer (64 dims each)")
print(f"   Hidden dimension     : 768")
print(f"   Total parameters     : {total_params:,}   (~110M BERT + ~4K head)")
print(f"   Trainable parameters : {trainable_params:,}  (all — full fine-tuning)")
print(f"   Classification head  : Linear(768 → {NUM_CLASSES} classes)")
print(f"   Device               : {device}")
print(f"\n   Memory estimate on GPU: ~{total_params * 4 / 1e9:.2f} GB (FP32)")

In [ ]:
# ── Load pre-trained BERT with classification head ─────────────────────
print(f"⏳ Loading {MODEL_NAME} for sequence classification...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    output_attentions=False,
    output_hidden_states=False
)
model = model.to(device)

# ── Model summary ──────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ BERT model loaded")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")
print(f"   Classification head  : 768 → {NUM_CLASSES} classes")
print(f"   Device               : {device}")

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────
EPOCHS        = 4
# WHY 4 epochs? BERT fine-tuning converges fast — the original BERT paper
# recommends 2–4 epochs. More epochs → overfitting on small datasets.

LEARNING_RATE = 2e-5
# WHY 2e-5? The BERT paper tested {1e-5, 2e-5, 3e-5, 5e-5} and found 2e-5
# consistently best for classification. Too high (5e-4) causes "catastrophic
# forgetting" — the pre-trained weights are destroyed. Too low is too slow.
# Rule of thumb for fine-tuning pre-trained Transformers: 1e-5 to 5e-5.

WARMUP_RATIO  = 0.1
# WHY warmup? At the start of fine-tuning, gradients are large and erratic
# (the new classification head is random). A warmup linearly ramps lr from
# ~0 → LEARNING_RATE over the first 10% of steps, preventing the pre-trained
# weights from being destabilised before the head stabilises.

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

# ── Optimizer & Scheduler ──────────────────────────────────────────────
# AdamW = Adam + Weight Decay (L2 regularisation on weights, NOT on biases/LayerNorm)
# eps=1e-8: numerical stability term in Adam's denominator — prevents division by ~zero
# when gradient variance is very small
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)

# Linear warmup → linear decay schedule
# Step 0 to warmup_steps : lr increases linearly from 0 to LEARNING_RATE
# warmup_steps to total_steps : lr decreases linearly from LEARNING_RATE to 0
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"⚙️  Training Configuration:")
print(f"   Epochs         : {EPOCHS}   (BERT paper recommendation: 2–4)")
print(f"   Batch size     : {BATCH_SIZE}")
print(f"   Learning rate  : {LEARNING_RATE}  (BERT paper sweet spot for fine-tuning)")
print(f"   Total steps    : {total_steps}")
print(f"   Warmup steps   : {warmup_steps}  ({WARMUP_RATIO:.0%} of total — stabilises early training)")
print(f"   Optimizer      : AdamW (Adam + decoupled weight decay)")
print(f"   LR Schedule    : Linear warmup → linear decay")

In [ ]:
# ── Helper: evaluate on any DataLoader ────────────────────────────────
def evaluate(model, loader):
    model.eval()   # switches off dropout + BatchNorm update — critical for consistent eval
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():   # disables gradient tracking → saves memory + speeds up eval
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            # When labels are passed, HF models compute CrossEntropyLoss internally
            # outputs.loss   = scalar loss
            # outputs.logits = raw scores (batch × num_classes) before softmax
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

            total_loss += outputs.loss.item()
            # argmax along class dimension → predicted class index
            preds       = torch.argmax(outputs.logits, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels


# ── Training loop ──────────────────────────────────────────────────────
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc  = 0.0
best_model_path = 'best_bert_healthcare.pt'

print("🚀 Starting Training...\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    # ── Training phase ─────────────────────────────────────────────────
    model.train()   # re-enable dropout (model.eval() was set in evaluate())
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        # ── CRITICAL: clear gradients BEFORE forward pass ──────────────
        # PyTorch accumulates gradients by default (+=, not =).
        # Without zero_grad(), each backward() adds to the previous batch's
        # gradients → effectively using a much larger (incorrect) batch size.
        # Always call before forward pass, not after.
        optimizer.zero_grad()

        # Forward pass — BERT runs self-attention across all 128 positions
        # labels passed → model computes CrossEntropyLoss(logits, labels) internally
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss   # scalar CrossEntropy loss for this batch

        # Backward pass — computes ∂loss/∂weight for every parameter via chain rule
        loss.backward()

        # ── Gradient clipping ──────────────────────────────────────────
        # Clips the global gradient norm to max_norm=1.0.
        # WHY? BERT has 110M params and 12 layers. Occasionally a bad batch
        # produces very large gradients (especially early in training when the
        # classification head is random). Without clipping, one bad step can
        # corrupt all the carefully pre-trained weights. The BERT paper and all
        # HF fine-tuning guides recommend max_norm=1.0.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update weights — applies the AdamW step using the clipped gradients
        optimizer.step()

        # Update learning rate — must come AFTER optimizer.step() (PyTorch convention).
        # scheduler.step() advances the warmup/decay curve by 1 step.
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs.logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # ── Validation phase ───────────────────────────────────────────────
    # Evaluated on held-out val set — tells us if we're overfitting
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    # ── Save best model ────────────────────────────────────────────────
    # Save only model state_dict (weights), not the full model object —
    # more portable and smaller file. Load back with model.load_state_dict().
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        saved_marker = " ✅ saved"
    else:
        saved_marker = ""

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>9.4f} | {val_acc:>7.2%}{saved_marker}")

print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.2%}")
print(f"\n💡 Expected pattern: train_loss ↓ each epoch, val_loss ↓ then plateaus (4 epochs is usually safe)")
print(f"   If val_loss starts rising while train_loss keeps falling → overfitting — reduce epochs.")

In [ ]:
# ── Helper: evaluate on any DataLoader ────────────────────────────────
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

            total_loss += outputs.loss.item()
            preds       = torch.argmax(outputs.logits, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels


# ── Training loop ──────────────────────────────────────────────────────
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc  = 0.0
best_model_path = 'best_bert_healthcare.pt'

print("🚀 Starting Training...\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>9} | {'Val Acc':>8}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    # ── Training phase ─────────────────────────────────────────────────
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        loss = outputs.loss
        loss.backward()

        # Gradient clipping — prevents exploding gradients in Transformers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs.logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # ── Validation phase ───────────────────────────────────────────────
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    # ── Save best model ────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        saved_marker = " ✅ saved"
    else:
        saved_marker = ""

    # ── Log ────────────────────────────────────────────────────────────
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>9.4f} | {val_acc:>7.2%}{saved_marker}")

print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.2%}")

## Step 8: Evaluate the Model

We evaluate using:
1. **Training & Validation curves** — check for overfitting
2. **Classification Report** — Precision, Recall, F1 per class
3. **Confusion Matrix** — which categories get confused with which

In [ ]:
# ── Plot training curves ───────────────────────────────────────────────
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
axes[0].plot(epochs_range, history['train_loss'], 'o-', color='#2196F3', label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'],   's--', color='#FF5722', label='Val Loss')
axes[0].set_title('Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_range, history['train_acc'], 'o-', color='#4CAF50', label='Train Acc')
axes[1].plot(epochs_range, history['val_acc'],   's--', color='#FF9800', label='Val Acc')
axes[1].set_title('Accuracy Curve', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('BERT Healthcare Classifier — Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Load best model and evaluate on test set ───────────────────────────
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader)

print(f"🧪 Test Set Results")
print(f"   Test Loss     : {test_loss:.4f}")
print(f"   Test Accuracy : {test_acc:.2%}")

# ── Classification Report ──────────────────────────────────────────────
class_names = le.classes_
print("\n📊 Classification Report:")
print(classification_report(test_labels, test_preds, target_names=class_names))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    linecolor='white'
)
plt.title('Confusion Matrix — BERT Healthcare Classifier', fontweight='bold', fontsize=13)
plt.ylabel('True Label', fontweight='bold')
plt.xlabel('Predicted Label', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("📌 Diagonal = correct predictions")
print("📌 Off-diagonal = misclassifications (which categories get confused)")

## Step 9: Predict on New Custom Text

Now the model is ready to classify **any new healthcare sentence** — this is how you'd deploy it in a real application.

```
New Text Input
     ↓  Tokenize
 [CLS] tokens [SEP]
     ↓  BERT Forward Pass
 Logits (5 values, one per class)
     ↓  Softmax
 Probabilities → Top prediction + confidence
```

In [ ]:
# ── Prediction function ────────────────────────────────────────────────
def predict(text, model, tokenizer, label_map, max_len=128):
    """
    Predict the healthcare category of a given text string.
    Returns predicted label, confidence score, and all class probabilities.
    """
    model.eval()
    encoding = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    probs      = torch.softmax(outputs.logits, dim=1).squeeze().cpu().numpy()
    pred_idx   = int(np.argmax(probs))
    pred_label = label_map[pred_idx]
    confidence = float(probs[pred_idx])

    return pred_label, confidence, {label_map[i]: round(float(p), 4) for i, p in enumerate(probs)}


# ── Test on new unseen sentences ───────────────────────────────────────
test_sentences = [
    "The patient was prescribed insulin therapy after being diagnosed with type 1 diabetes.",
    "MRI scan revealed a malignant tumor in the left lung requiring immediate chemotherapy.",
    "The doctor recommended a diet rich in leafy greens and omega-3 fatty acids.",
    "She was experiencing severe chest pain and shortness of breath indicating possible cardiac arrest.",
    "The therapist suggested cognitive behavioral therapy for managing panic attacks and depression.",
    "Blood glucose levels were dangerously high at 450 mg/dL requiring emergency insulin.",
]

print("🔮 Predictions on New Healthcare Texts")
print("=" * 75)

for text in test_sentences:
    label, conf, all_probs = predict(text, model, tokenizer, LABEL_MAP)
    short_text = text[:65] + "..." if len(text) > 65 else text
    print(f"\nText       : {short_text}")
    print(f"Prediction : {label:<20}  Confidence: {conf:.2%}")
    print(f"All probs  : ", end="")
    for cat, prob in sorted(all_probs.items(), key=lambda x: -x[1]):
        print(f"{cat}: {prob:.2%}", end="  ")
    print()

In [ ]:
# ── Interactive: try your own text ────────────────────────────────────
print("💬 Try your own healthcare text:")
print("   (Edit the text below and run the cell)\n")

my_text = "Radiation therapy was used to shrink the tumor before surgery."

label, conf, all_probs = predict(my_text, model, tokenizer, LABEL_MAP)

print(f"Input      : {my_text}")
print(f"Prediction : {label}")
print(f"Confidence : {conf:.2%}")
print("\nAll class probabilities:")

# Bar chart of probabilities
cats  = list(all_probs.keys())
probs = list(all_probs.values())
colors_bar = ['#E91E63' if c == label else '#90CAF9' for c in cats]

plt.figure(figsize=(8, 3))
bars = plt.barh(cats, probs, color=colors_bar, edgecolor='white')
for bar, val in zip(bars, probs):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.2%}', va='center', fontsize=10)
plt.xlim(0, 1.15)
plt.xlabel('Probability')
plt.title(f'BERT Prediction: "{my_text[:50]}..."', fontweight='bold')
plt.tight_layout()
plt.show()

## 🎓 Summary: What We Built

```
┌─────────────────────────────────────────────────────────────────┐
│         Healthcare BERT Text Classifier — Full Pipeline          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. DATA      → Scraped live from MedlinePlus (NIH)             │
│                 5 categories: Heart Disease, Diabetes,          │
│                 Mental Health, Cancer, Nutrition                │
│                                                                 │
│  2. TOKENIZE  → BertTokenizer → [CLS] tokens [SEP]             │
│                 input_ids + attention_mask                      │
│                                                                 │
│  3. MODEL     → bert-base-uncased (12 layers, 110M params)     │
│                 + Classification Head (768 → 5 classes)         │
│                                                                 │
│  4. TRAIN     → AdamW + Linear Warmup Scheduler                │
│                 4 epochs, batch=16, lr=2e-5                     │
│                                                                 │
│  5. EVALUATE  → Accuracy, F1, Confusion Matrix                 │
│                                                                 │
│  6. PREDICT   → Any new healthcare text → category + confidence│
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

### 🔑 Key Concepts Reinforced

| Concept | Where it appeared |
|---|---|
| **Tokenization + [CLS]/[SEP]** | Step 4 |
| **Attention mask** | Step 4 & 5 |
| **Pre-trained weights (Transfer Learning)** | Step 6 |
| **Fine-tuning** | Step 7 |
| **Gradient clipping** | Step 7 training loop |
| **Warmup + LR scheduling** | Step 7 |
| **[CLS] token for classification** | Step 6 architecture |
| **Confusion matrix / F1** | Step 8 |
| **Softmax → confidence score** | Step 9 |

---

### 🚀 Extensions for Learners

1. **Try DistilBERT** — replace `bert-base-uncased` with `distilbert-base-uncased` and compare speed vs accuracy
2. **Add more categories** — scrape additional topics (Respiratory, Orthopedics, Infectious Disease)
3. **Freeze BERT layers** — only train the classification head and compare performance
4. **Deploy with Gradio** — `pip install gradio` and wrap `predict()` in a web UI
5. **Try longer MAX_LEN** — change to 256 and observe training time vs accuracy tradeoff

---

### 📚 References

- **BERT Paper:** Devlin et al. (2018) — [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805)
- **Hugging Face Transformers:** https://huggingface.co/docs/transformers
- **Data Source:** MedlinePlus — https://medlineplus.gov (US National Library of Medicine)
- **Illustrated Transformer:** http://jalammar.github.io/illustrated-transformer/